# golangchain — Interactive Tour

A hands-on walkthrough of every component in the **golangchain** library,
the production-grade Go equivalent of LangChain + LangGraph.

**Kernel:** [GoNB](https://github.com/janpfeifer/gonb) (Go Jupyter kernel)  
**Module:** `github.com/golangchain/golangchain`

---

## Table of Contents

1. [Setup & Imports](#1-setup--imports)
2. [Schema — Messages, Documents & Tool Calls](#2-schema)
3. [Prompt Templates](#3-prompt-templates)
4. [Output Parsers](#4-output-parsers)
5. [Callbacks](#5-callbacks)
6. [LLM Interface & Mock LLM](#6-llm-interface--mock-llm)
7. [Chains (LCEL-style Runnables)](#7-chains--lcel-style-runnables)
8. [Memory](#8-memory)
9. [Tools](#9-tools)
10. [Agents — ReAct & ToolCalling](#10-agents)
11. [Vector Store & Embeddings](#11-vector-store--embeddings)
12. [StateGraph — LangGraph in Go](#12-stategraph)
13. [Human-in-the-Loop (Interrupt + Checkpoint)](#13-human-in-the-loop)
14. [Parallel Branches](#14-parallel-branches)
15. [Full Pipeline: Memory + Chain + Streaming](#15-full-pipeline)


---

## 1. Setup & Imports

GoNB cells that start with `%` are magic directives. We use `%goflags` to
point the kernel at our local module so every cell shares the same build cache.


In [2]:
// Tell GoNB where the module root lives.
// Adjust the path if you cloned golangchain elsewhere.
!*go work init && go work use . /notebooks/golangchain
%goworkfix
%goworkdir C:\Users\CC974EJ\OneDrive - EY\Documents\Software\Go\golangchain

go: /tmp/gonb_170140ce/go.work already exists
exit status 1


	- Replace rule for module "github.com/janpfeifer/gonb" to local directory "/home/jovyan/gonb" already exists.


"%goworkdir" unknown or not implemented yet.

In [ ]:
// Verify the module resolves cleanly.
import "fmt"
fmt.Println("GoNB + golangchain ready!")

---

## 2. Schema

The `schema` package is the single shared-type layer. Every other package
imports it; it imports nothing internal.

Key types:

- `Message` — a conversation turn (system / human / ai / tool)
- `Document` — a chunk of text with metadata, used in RAG
- `ToolCall` / `ToolDef` — native function-calling structures
- `Generation` — LLM response (text + message + usage)
- `StreamChunk` — one token during streaming


In [ ]:
import (
    "encoding/json"
    "fmt"

    "github.com/golangchain/golangchain/schema"
)

// --- Messages ---
sys   := schema.NewSystemMessage("You are a helpful assistant.")
human := schema.NewHumanMessage("What is the capital of France?")
ai    := schema.NewAIMessage("The capital of France is Paris.")
tool  := schema.NewToolMessage("42", "call-001", "calculator")

for _, m := range []schema.Message{sys, human, ai, tool} {
    fmt.Printf("role=%-8s content=%q\n", m.Role, m.Content)
}

// --- Document (RAG) ---
fmt.Println()
doc := schema.Document{
    PageContent: "Go was designed at Google by Robert Griesemer, Rob Pike, and Ken Thompson.",
    Metadata:    map[string]any{"source": "wikipedia", "year": 2009},
    Score:       0.95,
}
fmt.Printf("Document: %q (score=%.2f, source=%s)\n", doc.PageContent[:40]+"...", doc.Score, doc.Metadata["source"])

// --- ToolCall round-trip through JSON ---
fmt.Println()
tc := schema.ToolCall{
    ID:        "call-001",
    Type:      "function",
    Name:      "calculator",
    Arguments: json.RawMessage(`{"expression":"6 * 7"}`),
}
b, _ := json.Marshal(tc)
fmt.Printf("ToolCall JSON: %s\n", b)

---

## 3. Prompt Templates

golangchain uses Go's `text/template` syntax (`{{.VarName}}`) instead of
Python f-strings.

| Type                    | Use case                                             |
| ----------------------- | ---------------------------------------------------- |
| `PromptTemplate`        | Single string with variable substitution             |
| `ChatPromptTemplate`    | Ordered list of system/human/ai slots + placeholders |
| `FewShotPromptTemplate` | Auto-builds few-shot examples before the query       |


In [ ]:
import (
    "fmt"
    "github.com/golangchain/golangchain/prompt"
    "github.com/golangchain/golangchain/schema"
)

// --- 3a. PromptTemplate ---
pt := prompt.MustNewPromptTemplate(
    "Translate the following text into {{.Language}}: {{.Text}}",
)
result, _ := pt.Format(map[string]any{
    "Language": "French",
    "Text":     "Hello, world!",
})
fmt.Println("PromptTemplate:", result)

// --- 3b. ChatPromptTemplate with history placeholder ---
fmt.Println()
chat := prompt.MustNewChatPromptTemplate(
    prompt.MustSystem("You are an expert in {{.Topic}}."),
    prompt.NewMessagePlaceholder("history"),
    prompt.MustHuman("{{.Question}}"),
)

history := []schema.Message{
    schema.NewHumanMessage("What is a goroutine?"),
    schema.NewAIMessage("A goroutine is a lightweight thread managed by the Go runtime."),
}

msgs, _ := chat.FormatMessages(map[string]any{
    "Topic":    "Go programming",
    "history":  history,
    "Question": "How does a channel differ from a mutex?",
})

for i, m := range msgs {
    fmt.Printf("  [%d] role=%-8s %q\n", i, m.Role, m.Content)
}

// --- 3c. FewShotPromptTemplate ---
fmt.Println()
fst := &prompt.FewShotPromptTemplate{
    Prefix:        "Classify the sentiment of each review.",
    ExamplePrefix: "Review:",
    ExampleSuffix: "Sentiment:",
    Examples: []prompt.Example{
        {Input: "This product is amazing!",  Output: "positive"},
        {Input: "Terrible quality, avoid.",  Output: "negative"},
        {Input: "It's okay, nothing special.", Output: "neutral"},
    },
    SuffixTemplate: prompt.MustNewPromptTemplate("Review: {{.Review}}\nSentiment:"),
}
fewShot, _ := fst.Format(map[string]any{"Review": "Absolutely love it!"})
fmt.Println("FewShot prompt:\n" + fewShot)

---

## 4. Output Parsers

Output parsers transform raw LLM text into typed Go values. All implement
`Parser[T]` with a `Parse(string) (T, error)` method.

The `AsAny[T]` adapter bridges typed parsers to `chain.NewLLMChain`'s
untyped `interface{ Parse(string) (any, error) }`.


In [ ]:
import (
    "fmt"
    "github.com/golangchain/golangchain/output"
)

// --- 4a. StrOutputParser ---
str, _ := output.StrOutputParser{}.Parse("  Hello, GoNB!  ")
fmt.Println("Str:", str)

// --- 4b. JSONOutputParser (handles ```json fences) ---
jsonRaw := "```json\n{\"answer\": 42, \"unit\": \"km\"}\n```"
parsed, _ := output.JSONOutputParser{}.Parse(jsonRaw)
fmt.Printf("JSON: answer=%.0f unit=%s\n", parsed["answer"], parsed["unit"])

// --- 4c. StructOutputParser[T] ---
type WeatherReport struct {
    City   string  `json:"city"`
    TempC  float64 `json:"temp_c"`
    Cloudy bool    `json:"cloudy"`
}
structParser := output.NewStructOutputParser[WeatherReport]()
report, _ := structParser.Parse(`{"city":"Amsterdam","temp_c":14.5,"cloudy":true}`)
fmt.Printf("Struct: %s %.1f°C cloudy=%v\n", report.City, report.TempC, report.Cloudy)

// --- 4d. ListOutputParser ---
items, _ := output.NewListOutputParser(output.SepComma).Parse("Go, Python, Rust, TypeScript")
fmt.Printf("List (%d items): %v\n", len(items), items)

// --- 4e. BoolOutputParser ---
yes, _ := output.BoolOutputParser{}.Parse("yes")
no, _  := output.BoolOutputParser{}.Parse("false")
fmt.Printf("Bool: yes=%v, false=%v\n", yes, no)

// --- 4f. Format instructions ---
fmt.Println("\nFormat instructions:")
fmt.Println(" Str:",  output.StrOutputParser{}.FormatInstructions())
fmt.Println(" JSON:", output.JSONOutputParser{}.FormatInstructions())
fmt.Println(" Bool:", output.BoolOutputParser{}.FormatInstructions())

---

## 5. Callbacks

`callbacks.Handler` is a 13-method interface emitted by every component.
Embed `NoOpHandler` and override only what you need. `CallbackManager`
fans events out to multiple handlers concurrently.


In [ ]:
import (
    "context"
    "fmt"
    "strings"

    "github.com/golangchain/golangchain/callbacks"
    "github.com/golangchain/golangchain/schema"
)

// --- 5a. Custom handler that collects events into a log ---
type eventLog struct {
    callbacks.NoOpHandler
    lines []string
}

func (e *eventLog) OnLLMStart(_ context.Context, model string, msgs []schema.Message) {
    e.lines = append(e.lines, fmt.Sprintf("LLMStart  model=%s msgs=%d", model, len(msgs)))
}
func (e *eventLog) OnLLMEnd(_ context.Context, model string, gen *schema.Generation) {
    e.lines = append(e.lines, fmt.Sprintf("LLMEnd    model=%s text=%q", model, gen.Text))
}
func (e *eventLog) OnChainStart(_ context.Context, name string, _ map[string]any) {
    e.lines = append(e.lines, fmt.Sprintf("ChainStart name=%s", name))
}
func (e *eventLog) OnChainEnd(_ context.Context, name string, _ map[string]any) {
    e.lines = append(e.lines, fmt.Sprintf("ChainEnd   name=%s", name))
}
func (e *eventLog) OnToolStart(_ context.Context, tool, input string) {
    e.lines = append(e.lines, fmt.Sprintf("ToolStart  tool=%s input=%q", tool, input))
}
func (e *eventLog) OnToolEnd(_ context.Context, tool, out string) {
    e.lines = append(e.lines, fmt.Sprintf("ToolEnd    tool=%s out=%q", tool, out))
}

log1 := &eventLog{}
log2 := &eventLog{} // second handler — fan-out demo

// --- 5b. CallbackManager ---
cm := callbacks.NewCallbackManager(log1)
cm.Add(log2) // add at runtime

ctx := context.Background()
fakeGen := &schema.Generation{Text: "Paris", StopReason: "stop"}

cm.OnChainStart(ctx, "MyChain", map[string]any{"q": "capital?"})
cm.OnLLMStart(ctx, "gpt-4o", []schema.Message{schema.NewHumanMessage("capital?")})
cm.OnLLMEnd(ctx, "gpt-4o", fakeGen)
cm.OnToolStart(ctx, "calculator", "6*7")
cm.OnToolEnd(ctx, "calculator", "42")
cm.OnChainEnd(ctx, "MyChain", map[string]any{"output": "Paris"})

fmt.Println("Handler 1 log:")
fmt.Println(strings.Join(log1.lines, "\n"))
fmt.Printf("\nHandler 2 received same %d events\n", len(log2.lines))

// --- 5c. Built-in LoggingHandler ---
fmt.Println("\n--- LoggingHandler output ---")
lcm := callbacks.NewCallbackManager(callbacks.NewLoggingHandler(func(f string, a ...any) {
    fmt.Printf(f+"\n", a...)
}))
lcm.OnLLMStart(ctx, "gpt-4o-mini", []schema.Message{human})
lcm.OnLLMEnd(ctx, "gpt-4o-mini", fakeGen)

---

## 6. LLM Interface & Mock LLM

All examples from here onward use a **mock LLM** so no API key is required.
In production, swap it with `openai.New(...)`, `anthropic.New(...)`, etc.


In [ ]:
import (
    "context"
    "encoding/json"
    "fmt"

    "github.com/golangchain/golangchain/llm"
    "github.com/golangchain/golangchain/schema"
)

// mockLLM satisfies llm.LLM without any network calls.
// It returns a fixed response, optionally with tool calls.
type mockLLM struct {
    response  string
    toolCalls []schema.ToolCall
    callCount int
}

func (m *mockLLM) ModelName() string { return "mock-llm" }

func (m *mockLLM) Generate(_ context.Context, msgs []schema.Message, opts ...llm.Option) (*schema.Generation, error) {
    m.callCount++
    msg := schema.Message{
        Role:      schema.RoleAI,
        Content:   m.response,
        ToolCalls: m.toolCalls,
    }
    return &schema.Generation{
        Text:    m.response,
        Message: msg,
        Usage:   schema.TokenUsage{PromptTokens: len(msgs) * 10, CompletionTokens: 20, TotalTokens: len(msgs)*10 + 20},
    }, nil
}

func (m *mockLLM) Stream(ctx context.Context, msgs []schema.Message, opts ...llm.Option) (<-chan schema.StreamChunk, error) {
    ch := make(chan schema.StreamChunk, 4)
    go func() {
        defer close(ch)
        words := []string{}
        cur := ""
        for _, r := range m.response + " " {
            if r == ' ' && cur != "" {
                words = append(words, cur+" ")
                cur = ""
            } else if r != ' ' {
                cur += string(r)
            }
        }
        for i, w := range words {
            ch <- schema.StreamChunk{Text: w, Done: i == len(words)-1}
        }
    }()
    return ch, nil
}

// Verify it satisfies the interface.
var _ llm.LLM = (*mockLLM)(nil)

// Quick smoke test.
mock := &mockLLM{response: "Paris is the capital of France."}
gen, _ := mock.Generate(context.Background(), []schema.Message{
    schema.NewHumanMessage("What is the capital of France?"),
})
fmt.Printf("Generate: %q\n", gen.Text)
fmt.Printf("Usage: prompt=%d completion=%d total=%d\n",
    gen.Usage.PromptTokens, gen.Usage.CompletionTokens, gen.Usage.TotalTokens)

// Stream demo
fmt.Print("Stream: ")
ch, _ := mock.Stream(context.Background(), nil)
for chunk := range ch {
    fmt.Print(chunk.Text)
}
fmt.Println()

// Tool-calling variant
toolMock := &mockLLM{
    response: "",
    toolCalls: []schema.ToolCall{
        {
            ID:        "call-1",
            Type:      "function",
            Name:      "calculator",
            Arguments: json.RawMessage(`{"expression":"6*7"}`),
        },
    },
}
gen2, _ := toolMock.Generate(context.Background(), nil)
fmt.Printf("\nToolCall mock: %d tool calls, first=%s args=%s\n",
    len(gen2.Message.ToolCalls),
    gen2.Message.ToolCalls[0].Name,
    gen2.Message.ToolCalls[0].Arguments)

---

## 7. Chains (LCEL-style Runnables)

Every chain component implements `Runnable` — a composable interface with
`Invoke`, `Stream`, and `Pipe`.

| Chain             | Purpose                                      |
| ----------------- | -------------------------------------------- |
| `FuncRunnable`    | Wrap any `func(ctx, any) (any, error)`       |
| `LLMRunnable`     | Wrap an `llm.LLM` for use in a Pipe          |
| `LLMChain`        | `prompt → LLM → parser` in one object        |
| `SequentialChain` | Thread output of step N as input to step N+1 |
| `MapChain`        | Fan input to multiple branches in parallel   |
| `RouterChain`     | Pick one branch based on a routing function  |


In [ ]:
// --- 7a. FuncRunnable & Pipe ---
import (
    "context"
    "fmt"
    "strings"

    "github.com/golangchain/golangchain/chain"
)

upper := chain.NewFuncRunnable("upper", func(_ context.Context, in any) (any, error) {
    return strings.ToUpper(in.(string)), nil
})
exclaim := chain.NewFuncRunnable("exclaim", func(_ context.Context, in any) (any, error) {
    return in.(string) + "!!!", nil
})

pipe := upper.Pipe(exclaim)
result, _ := pipe.Invoke(context.Background(), "hello world")
fmt.Println("Pipe result:", result)

// Stream through a pipe
fmt.Print("Pipe stream: ")
ch, _ := pipe.Stream(context.Background(), "hello world")
for chunk := range ch {
    if chunk.Err == nil {
        fmt.Print(chunk.Value)
    }
}
fmt.Println()

In [ ]:
// --- 7b. LLMChain: prompt → LLM → StrOutputParser ---
import (
    "context"
    "fmt"

    "github.com/golangchain/golangchain/chain"
    "github.com/golangchain/golangchain/output"
    "github.com/golangchain/golangchain/prompt"
)

llmModel := &mockLLM{response: "The capital of Germany is Berlin."}

chatPrompt := prompt.MustNewChatPromptTemplate(
    prompt.MustSystem("You are a geography expert. Answer concisely."),
    prompt.MustHuman("What is the capital of {{.Country}}?"),
)

c := chain.NewLLMChain(
    chatPrompt,
    llmModel,
    output.AsAny(output.StrOutputParser{}),
    chain.WithChainName("GeoChain"),
)

ans, _ := c.Invoke(context.Background(), map[string]any{"Country": "Germany"})
fmt.Println("LLMChain answer:", ans)

// Streaming
fmt.Print("LLMChain stream: ")
streamCh, _ := c.Stream(context.Background(), map[string]any{"Country": "Germany"})
for chunk := range streamCh {
    if chunk.Err == nil && !chunk.Done {
        fmt.Print(chunk.Value)
    }
}
fmt.Println()

In [ ]:
// --- 7c. SequentialChain ---
import (
    "context"
    "fmt"
    "strings"

    "github.com/golangchain/golangchain/chain"
)

step1 := chain.NewFuncRunnable("tokenise", func(_ context.Context, in any) (any, error) {
    return strings.Fields(in.(string)), nil
})
step2 := chain.NewFuncRunnable("count", func(_ context.Context, in any) (any, error) {
    words := in.([]string)
    return fmt.Sprintf("%d words", len(words)), nil
})
step3 := chain.NewFuncRunnable("shout", func(_ context.Context, in any) (any, error) {
    return strings.ToUpper(in.(string)), nil
})

seq := chain.NewSequentialChain("WordCounter", step1, step2, step3)
out, _ := seq.Invoke(context.Background(), "The quick brown fox jumps over the lazy dog")
fmt.Println("SequentialChain:", out)

In [ ]:
// --- 7d. MapChain: parallel branches ---
import (
    "context"
    "fmt"
    "strings"
    "unicode/utf8"

    "github.com/golangchain/golangchain/chain"
)

branches := map[string]chain.Runnable{
    "upper":      chain.NewFuncRunnable("upper", func(_ context.Context, in any) (any, error) {
        return strings.ToUpper(in.(string)), nil
    }),
    "word_count": chain.NewFuncRunnable("wc", func(_ context.Context, in any) (any, error) {
        return len(strings.Fields(in.(string))), nil
    }),
    "char_count": chain.NewFuncRunnable("cc", func(_ context.Context, in any) (any, error) {
        return utf8.RuneCountInString(in.(string)), nil
    }),
}

mc := chain.NewMapChain("TextAnalyser", branches)
result, _ := mc.Invoke(context.Background(), "Hello, golangchain!")
m := result.(map[string]any)
fmt.Printf("MapChain results:\n  upper=%s\n  words=%v\n  chars=%v\n",
    m["upper"], m["word_count"], m["char_count"])

In [ ]:
// --- 7e. RouterChain ---
import (
    "context"
    "fmt"
    "strings"

    "github.com/golangchain/golangchain/chain"
)

mathChain    := chain.NewFuncRunnable("math",    func(_ context.Context, in any) (any, error) { return "[MATH] " + in.(string), nil })
scienceChain := chain.NewFuncRunnable("science", func(_ context.Context, in any) (any, error) { return "[SCIENCE] " + in.(string), nil })
defaultChain := chain.NewFuncRunnable("default", func(_ context.Context, in any) (any, error) { return "[GENERAL] " + in.(string), nil })

router := chain.NewRouterChain(
    "TopicRouter",
    func(_ context.Context, in any) (string, error) {
        s := strings.ToLower(in.(string))
        switch {
        case strings.Contains(s, "math") || strings.Contains(s, "calculat"):
            return "math", nil
        case strings.Contains(s, "physics") || strings.Contains(s, "biology"):
            return "science", nil
        default:
            return "unknown", nil
        }
    },
    map[string]chain.Runnable{
        "math":    mathChain,
        "science": scienceChain,
    },
    defaultChain,
)

for _, q := range []string{
    "Calculate 2+2",
    "Explain quantum physics",
    "What is Go?",
} {
    out, _ := router.Invoke(context.Background(), q)
    fmt.Printf("Q: %-30s → %s\n", q, out)
}

---

## 8. Memory

Three memory implementations share the same `Memory` interface:

| Type                        | Behaviour                            |
| --------------------------- | ------------------------------------ |
| `ConversationBufferMemory`  | Full history, unbounded              |
| `ConversationWindowMemory`  | Sliding window of last _k_ turns     |
| `ConversationSummaryMemory` | Compresses old turns via an LLM call |


In [ ]:
import (
    "context"
    "fmt"

    "github.com/golangchain/golangchain/memory"
    "github.com/golangchain/golangchain/schema"
)

ctx := context.Background()

// --- 8a. ConversationBufferMemory ---
buf := memory.NewConversationBufferMemory()
_ = buf.SaveContext(ctx, "Hi!",                    "Hello! How can I help?")
_ = buf.SaveContext(ctx, "What is Go?",             "Go is a statically typed compiled language.")
_ = buf.SaveContext(ctx, "Who created it?",         "Go was created at Google.")

vars, _ := buf.LoadMemoryVariables(ctx)
history := vars["history"].([]schema.Message)
fmt.Printf("Buffer: %d messages stored\n", len(history))
for _, m := range history {
    fmt.Printf("  %-8s %s\n", m.Role, m.Content)
}

// --- 8b. ConversationWindowMemory (k=2) ---
fmt.Println()
win := memory.NewConversationWindowMemory(2) // keep last 2 turns
for _, turn := range [][2]string{
    {"Turn 1 question", "Turn 1 answer"},
    {"Turn 2 question", "Turn 2 answer"},
    {"Turn 3 question", "Turn 3 answer"},
    {"Turn 4 question", "Turn 4 answer"},
} {
    _ = win.SaveContext(ctx, turn[0], turn[1])
}
wVars, _ := win.LoadMemoryVariables(ctx)
wHistory := wVars["history"].([]schema.Message)
fmt.Printf("Window (k=2): %d messages visible (discarded 2 oldest turns)\n", len(wHistory))
for _, m := range wHistory {
    fmt.Printf("  %-8s %s\n", m.Role, m.Content)
}

// --- 8c. ConversationSummaryMemory (mock LLM summarises) ---
fmt.Println()
summaryLLM := &mockLLM{response: "The user asked about Go language basics and its origins at Google."}
sum := memory.NewConversationSummaryMemory(summaryLLM)
sum.MaxRecent = 2 // compress after 2 turns

_ = sum.SaveContext(ctx, "What is Go?",          "A statically typed compiled language.")
_ = sum.SaveContext(ctx, "Who made it?",          "Google.")
// This third save triggers compression (MaxRecent=2 => threshold=4 messages)
_ = sum.SaveContext(ctx, "When was it released?", "In 2009.")
_ = sum.SaveContext(ctx, "Is it fast?",           "Yes, comparable to C.")
_ = sum.SaveContext(ctx, "Trigger summary now",   "Compressing...")

msgs := sum.Messages()
fmt.Printf("Summary memory: %d messages (first is compressed summary)\n", len(msgs))
for _, m := range msgs {
    fmt.Printf("  %-8s %s\n", m.Role, m.Content)
}

---

## 9. Tools

Every tool implements `Tool`: `Name()`, `Description()`, `Schema()`, `Run(ctx, input)`.

Built-in tools:

- `Calculator` — recursive-descent parser (no `eval`, pure Go)
- `HTTPFetch` — GET any URL, returns body as string
- `DuckDuckGoSearch` — Instant Answer API (no key)
- `ShellTool` — runs shell commands
- `FuncTool` — wraps any function


In [ ]:
import (
    "context"
    "encoding/json"
    "fmt"

    "github.com/golangchain/golangchain/tools"
)

ctx := context.Background()

// --- 9a. Calculator ---
calc := tools.Calculator{}
fmt.Printf("Calculator: %s\n", calc.Description())

exprs := []string{
    "2 + 2",
    "10 * (3 + 4)",
    "2 ^ 10",
    "sqrt(144)",
    "abs(-99)",
    "floor(3.9)",
    `{"expression":"(1 + 2) * 3"}`, // JSON input (as sent by tool-calling agents)
}
for _, e := range exprs {
    r, _ := calc.Run(ctx, e)
    fmt.Printf("  %-30s = %s\n", e, r)
}

// --- 9b. FuncTool ---
fmt.Println()
wordCountTool := tools.NewFuncTool(
    "word_count",
    "Counts the number of words in the input text.",
    json.RawMessage(`{"type":"object","properties":{"text":{"type":"string"}},"required":["text"]})`),
    func(_ context.Context, input string) (string, error) {
        var args struct{ Text string `json:"text"` }
        if err := json.Unmarshal([]byte(input), &args); err != nil {
            return fmt.Sprintf("%d", len(input)), nil
        }
        count := len(splitWords(args.Text))
        return fmt.Sprintf("%d words", count), nil
    },
)

r, _ := wordCountTool.Run(ctx, `{"text":"The quick brown fox jumps over the lazy dog"}`)
fmt.Println("FuncTool word_count:", r)

// --- 9c. ToToolDef / FindTool ---
fmt.Println()
allTools := []tools.Tool{calc, wordCountTool}
defs := tools.ToToolDefs(allTools)
for _, d := range defs {
    fmt.Printf("ToolDef: %-15s %s\n", d.Name, d.Description[:40]+"...")
}
found := tools.FindTool(allTools, "calculator")
fmt.Printf("\nFindTool('calculator') → %s\n", found.Name())

// helper used by the FuncTool above
func splitWords(s string) []string {
    var words []string
    cur := ""
    for _, r := range s + " " {
        if r == ' ' || r == '\t' || r == '\n' {
            if cur != "" { words = append(words, cur); cur = "" }
        } else {
            cur += string(r)
        }
    }
    return words
}

---

## 10. Agents

Two agent strategies, one `AgentExecutor`:

| Strategy           | How it works                                                 |
| ------------------ | ------------------------------------------------------------ |
| `ReActAgent`       | Parses `Thought/Action/Action Input/Final Answer` text loops |
| `ToolCallingAgent` | Uses native tool-call API (`gen.Message.ToolCalls`)          |

`AgentExecutor.Run` blocks; `AgentExecutor.Stream` yields `AgentEvent` values in real time.


In [ ]:
// --- 10a. ReActAgent with text-based loop ---
import (
    "context"
    "fmt"

    "github.com/golangchain/golangchain/agent"
    "github.com/golangchain/golangchain/tools"
)

// Mock that returns a ReAct-formatted response on first call, final answer on second.
type reactMock struct{ calls int }

func (r *reactMock) ModelName() string { return "react-mock" }
func (r *reactMock) Generate(_ context.Context, _ []schema.Message, _ ...llm.Option) (*schema.Generation, error) {
    r.calls++
    if r.calls == 1 {
        return &schema.Generation{Text: "Thought: I should calculate this.\nAction: calculator\nAction Input: 6 * 7"}, nil
    }
    return &schema.Generation{Text: "Final Answer: 6 * 7 = 42"}, nil
}
func (r *reactMock) Stream(_ context.Context, _ []schema.Message, _ ...llm.Option) (<-chan schema.StreamChunk, error) {
    ch := make(chan schema.StreamChunk, 1)
    gen, _ := r.Generate(context.Background(), nil)
    ch <- schema.StreamChunk{Text: gen.Text, Done: true}
    close(ch)
    return ch, nil
}

var _ llm.LLM = (*reactMock)(nil)

agentTools := []tools.Tool{tools.Calculator{}}
reactAgent := agent.NewReActAgent(&reactMock{}, agentTools)
executor   := agent.NewAgentExecutor(reactAgent, agentTools, agent.WithMaxIter(5))

answer, _ := executor.Run(context.Background(), "What is 6 * 7?")
fmt.Println("ReAct final answer:", answer)

In [ ]:
// --- 10b. ReActAgent streaming events ---
import (
    "context"
    "fmt"

    "github.com/golangchain/golangchain/agent"
    "github.com/golangchain/golangchain/tools"
)

agentTools2 := []tools.Tool{tools.Calculator{}}
reactAgent2 := agent.NewReActAgent(&reactMock{}, agentTools2)
executor2   := agent.NewAgentExecutor(reactAgent2, agentTools2, agent.WithMaxIter(5))

fmt.Println("Streaming agent events:")
for event := range executor2.Stream(context.Background(), "What is 6 * 7?") {
    switch event.Type {
    case agent.EventThought:
        fmt.Println("  [Thought]     ", event.Thought)
    case agent.EventToolCall:
        fmt.Printf("  [ToolCall]     %s(%s)\n", event.Action.Tool, event.Action.ToolInput)
    case agent.EventToolResult:
        fmt.Println("  [Observation] ", event.Observation)
    case agent.EventFinalAnswer:
        fmt.Println("  [Answer]      ", event.Answer)
    case agent.EventError:
        fmt.Println("  [Error]       ", event.Err)
    }
}

In [ ]:
// --- 10c. ToolCallingAgent (native function-calling) ---
import (
    "context"
    "encoding/json"
    "fmt"

    "github.com/golangchain/golangchain/agent"
    "github.com/golangchain/golangchain/schema"
    "github.com/golangchain/golangchain/tools"
)

// Mock that emits a tool call on first call, final text on second.
type toolCallMock struct{ calls int }

func (t *toolCallMock) ModelName() string { return "tool-call-mock" }
func (t *toolCallMock) Generate(_ context.Context, _ []schema.Message, _ ...llm.Option) (*schema.Generation, error) {
    t.calls++
    if t.calls == 1 {
        return &schema.Generation{
            Message: schema.Message{
                Role: schema.RoleAI,
                ToolCalls: []schema.ToolCall{{
                    ID: "c1", Name: "calculator",
                    Arguments: json.RawMessage(`{"expression":"1337 * 42"}`),
                }},
            },
        }, nil
    }
    return &schema.Generation{Text: "1337 * 42 = 56154"}, nil
}
func (t *toolCallMock) Stream(_ context.Context, _ []schema.Message, _ ...llm.Option) (<-chan schema.StreamChunk, error) {
    ch := make(chan schema.StreamChunk, 1)
    gen, _ := t.Generate(context.Background(), nil)
    ch <- schema.StreamChunk{Text: gen.Text, Done: true}
    close(ch)
    return ch, nil
}
var _ llm.LLM = (*toolCallMock)(nil)

tcTools  := []tools.Tool{tools.Calculator{}}
tcAgent  := agent.NewToolCallingAgent(&toolCallMock{}, tcTools, "You are a math assistant.")
tcExec   := agent.NewAgentExecutor(tcAgent, tcTools, agent.WithMaxIter(5))

fmt.Println("ToolCallingAgent events:")
for event := range tcExec.Stream(context.Background(), "What is 1337 * 42?") {
    switch event.Type {
    case agent.EventToolCall:
        fmt.Printf("  [ToolCall]     %s %s\n", event.Action.Tool, event.Action.ToolInput)
    case agent.EventToolResult:
        fmt.Println("  [Observation] ", event.Observation)
    case agent.EventFinalAnswer:
        fmt.Println("  [Answer]      ", event.Answer)
    }
}

---

## 11. Vector Store & Embeddings

`InMemoryVectorStore` indexes documents by cosine similarity.  
The `Embedder` interface is satisfied by `OpenAIEmbedder`, `AzureEmbedder`, or a mock.


In [ ]:
import (
    "context"
    "fmt"

    "github.com/golangchain/golangchain/embeddings"
    "github.com/golangchain/golangchain/schema"
    "github.com/golangchain/golangchain/vectorstore"
)

// mockEmbedder turns text into a simple bag-of-char-codes vector.
// In production use embeddings.NewOpenAIEmbedder(...).
type mockEmbedder struct{}

func (mockEmbedder) EmbedDocuments(_ context.Context, texts []string) ([][]float64, error) {
    result := make([][]float64, len(texts))
    for i, t := range texts {
        result[i] = textVector(t)
    }
    return result, nil
}
func (mockEmbedder) EmbedQuery(_ context.Context, text string) ([]float64, error) {
    return textVector(text), nil
}
var _ embeddings.Embedder = mockEmbedder{}

// Very naive 8-dim vector: normalised char-frequency for letters a-h.
func textVector(s string) []float64 {
    v := make([]float64, 8)
    for _, r := range s {
        idx := int(r-'a') % 8
        if idx >= 0 && idx < 8 {
            v[idx]++
        }
    }
    // L2 normalise
    var sum float64
    for _, x := range v { sum += x * x }
    if sum > 0 {
        for i := range v { v[i] /= sum }
    }
    return v
}

ctx  := context.Background()
store := vectorstore.NewInMemoryVectorStore(mockEmbedder{})

docs := []schema.Document{
    {PageContent: "Go is a statically typed compiled language designed at Google.",       Metadata: map[string]any{"id": "doc1", "topic": "Go"}},
    {PageContent: "Python is an interpreted high-level programming language.",             Metadata: map[string]any{"id": "doc2", "topic": "Python"}},
    {PageContent: "Rust guarantees memory safety without a garbage collector.",            Metadata: map[string]any{"id": "doc3", "topic": "Rust"}},
    {PageContent: "Go's goroutines make concurrent programming simple and efficient.",     Metadata: map[string]any{"id": "doc4", "topic": "Go"}},
    {PageContent: "LangChain provides composable building blocks for LLM applications.",  Metadata: map[string]any{"id": "doc5", "topic": "LLM"}},
}
_ = store.AddDocuments(ctx, docs)
fmt.Printf("Indexed %d documents\n", store.Len())

// Similarity search
results, _ := store.SimilaritySearch(ctx, "goroutine concurrency golang", 3)
fmt.Println("\nTop-3 results for 'goroutine concurrency golang':")
for i, d := range results {
    fmt.Printf("  [%d] score=%.4f  %s\n", i+1, d.Score, d.PageContent[:50]+"...")
}

// Delete by metadata ID
_ = store.Delete(ctx, []string{"doc1"})
fmt.Printf("\nAfter deleting doc1: %d documents remain\n", store.Len())

---

## 12. StateGraph

The `graph` package is the Go equivalent of **LangGraph**.

```
StateGraph[S]
  ├── AddNode(name, NodeFunc[S])           — register a processing node
  ├── AddEdge(from, to)                    — unconditional transition
  ├── AddConditionalEdges(from, fn, map)   — dynamic routing
  ├── AddParallelEdges(from, []string)     — fan-out concurrency
  └── Compile(opts...) → CompiledGraph[S]
        ├── Invoke(ctx, input, opts...)    — blocking
        └── Stream(ctx, input, opts...)   — <-chan GraphEvent[S]
```

The generic `StateReducer[S]` merges partial updates from each node into
the shared state — analogous to Redux reducers.


In [ ]:
// --- 12a. Linear graph: A → B → C → END ---
import (
    "context"
    "fmt"

    "github.com/golangchain/golangchain/graph"
)

type PipelineState struct {
    Value int
    Steps []string
}

reducer := func(cur, upd PipelineState) PipelineState {
    cur.Value += upd.Value
    cur.Steps  = append(cur.Steps, upd.Steps...)
    return cur
}

g := graph.NewStateGraph(reducer).WithName("LinearPipeline")

g.MustAddNode("double", func(_ context.Context, s PipelineState) (PipelineState, error) {
    return PipelineState{Value: s.Value, Steps: []string{"doubled"}}, nil
})
g.MustAddNode("add_ten", func(_ context.Context, s PipelineState) (PipelineState, error) {
    return PipelineState{Value: 10, Steps: []string{"added 10"}}, nil
})
g.MustAddNode("stringify", func(_ context.Context, s PipelineState) (PipelineState, error) {
    return PipelineState{Steps: []string{fmt.Sprintf("final=%d", s.Value)}}, nil
})

g.MustAddEdge(graph.START, "double")
g.MustAddEdge("double",    "add_ten")
g.MustAddEdge("add_ten",   "stringify")
g.MustAddEdge("stringify",  graph.END)

compiled, _ := g.Compile()
result, _ := compiled.Invoke(context.Background(), PipelineState{Value: 5})

fmt.Printf("Linear graph result: value=%d steps=%v\n", result.Value, result.Steps)

In [ ]:
// --- 12b. Conditional edges + streaming GraphEvents ---
import (
    "context"
    "fmt"

    "github.com/golangchain/golangchain/graph"
)

type RouterState struct {
    Input  string
    Output string
    Route  string
}

rg := graph.NewStateGraph(func(cur, upd RouterState) RouterState {
    if upd.Output != "" { cur.Output = upd.Output }
    if upd.Route  != "" { cur.Route  = upd.Route }
    return cur
})

rg.MustAddNode("classify", func(_ context.Context, s RouterState) (RouterState, error) {
    route := "general"
    if len(s.Input) < 10 { route = "short" } else { route = "long" }
    return RouterState{Route: route}, nil
})
rg.MustAddNode("short_handler", func(_ context.Context, s RouterState) (RouterState, error) {
    return RouterState{Output: "[SHORT] " + s.Input}, nil
})
rg.MustAddNode("long_handler", func(_ context.Context, s RouterState) (RouterState, error) {
    return RouterState{Output: "[LONG] " + s.Input}, nil
})

rg.MustAddEdge(graph.START, "classify")
rg.MustAddConditionalEdges("classify",
    func(_ context.Context, s RouterState) string { return s.Route },
    map[string]string{
        "short": "short_handler",
        "long":  "long_handler",
        "*":     "long_handler", // wildcard fallback
    },
)
rg.MustAddEdge("short_handler", graph.END)
rg.MustAddEdge("long_handler",  graph.END)

rc, _ := rg.Compile()

for _, input := range []string{"Hi!", "This is a much longer input string for routing."} {
    fmt.Printf("\nInput: %q\n", input)
    for ev := range rc.Stream(context.Background(), RouterState{Input: input}) {
        switch ev.Type {
        case graph.GraphEventNodeEnd:
            fmt.Printf("  [NodeEnd]   %-14s route=%q output=%q\n",
                ev.Node, ev.State.Route, ev.State.Output)
        case graph.GraphEventEnd:
            fmt.Printf("  [Done]      output=%q\n", ev.State.Output)
        }
    }
}

In [ ]:
// --- 12c. Cyclic graph: agent loop (agent → tools → agent → END) ---
import (
    "context"
    "fmt"

    "github.com/golangchain/golangchain/graph"
    "github.com/golangchain/golangchain/tools"
)

type AgentLoopState struct {
    Query   string
    Result  string
    Expr    string   // expression to calculate
    Next    string   // "tools" or "end"
    Iters   int
}

lg := graph.NewStateGraph(func(cur, upd AgentLoopState) AgentLoopState {
    if upd.Result != "" { cur.Result = upd.Result }
    if upd.Expr   != "" { cur.Expr   = upd.Expr }
    if upd.Next   != "" { cur.Next   = upd.Next }
    cur.Iters += upd.Iters
    return cur
})

lg.MustAddNode("agent", func(_ context.Context, s AgentLoopState) (AgentLoopState, error) {
    if s.Result != "" {
        // We have a tool result — produce final answer
        return AgentLoopState{Result: "Answer: " + s.Result, Next: "end"}, nil
    }
    // Decide to call a tool
    return AgentLoopState{Expr: "(42 + 58) * 2", Next: "tools", Iters: 1}, nil
})

lg.MustAddNode("tools", func(_ context.Context, s AgentLoopState) (AgentLoopState, error) {
    result, _ := tools.Calculator{}.Run(context.Background(), s.Expr)
    return AgentLoopState{Result: result, Iters: 1}, nil
})

lg.MustAddEdge(graph.START, "agent")
lg.MustAddConditionalEdges("agent",
    func(_ context.Context, s AgentLoopState) string { return s.Next },
    map[string]string{"tools": "tools", "end": graph.END},
)
lg.MustAddEdge("tools", "agent")

lc, _ := lg.Compile(graph.WithMaxSteps[AgentLoopState](10))
final, _ := lc.Invoke(context.Background(), AgentLoopState{Query: "What is (42+58)*2?"})
fmt.Printf("Cyclic agent loop: %s  (iterations=%d)\n", final.Result, final.Iters)

---

## 13. Human-in-the-Loop (Interrupt + Checkpoint)

A node returns `graph.NewInterrupt(message)` to **pause** execution.
The `MemoryCheckpointer` saves state. The caller can inspect, modify, and
**resume** by invoking the compiled graph again with the saved state.


In [ ]:
import (
    "context"
    "fmt"

    "github.com/golangchain/golangchain/graph"
)

type ApprovalState struct {
    Doc      string
    Approved bool
    Result   string
}

reducer := func(cur, upd ApprovalState) ApprovalState {
    if upd.Doc      != ""    { cur.Doc      = upd.Doc }
    if upd.Result   != ""    { cur.Result   = upd.Result }
    if upd.Approved          { cur.Approved = true }
    return cur
}

ag := graph.NewStateGraph(reducer)

ag.MustAddNode("review", func(_ context.Context, s ApprovalState) (ApprovalState, error) {
    fmt.Printf("  [review] Reviewing doc: %q — requesting human approval...\n", s.Doc)
    return s, graph.NewInterrupt("Human must approve before publishing")
})
ag.MustAddNode("publish", func(_ context.Context, s ApprovalState) (ApprovalState, error) {
    return ApprovalState{Result: "Published: " + s.Doc}, nil
})

ag.MustAddEdge(graph.START, "review")
ag.MustAddEdge("review",    "publish")
ag.MustAddEdge("publish",    graph.END)

checkpointer := graph.NewMemoryCheckpointer[ApprovalState]()
compiled, _  := ag.Compile(
    graph.WithCheckpointer[ApprovalState](checkpointer),
)

ctx      := context.Background()
threadID := "approval-thread-1"
initial  := ApprovalState{Doc: "golangchain release notes v1.0"}

// First run — will pause at Interrupt
_, err := compiled.Invoke(ctx, initial, graph.WithThreadID[ApprovalState](threadID))
fmt.Printf("\nRun 1 paused: %v\n", err)

// List checkpoints
history, _ := checkpointer.List(ctx, threadID)
fmt.Printf("Saved %d checkpoint(s)\n", len(history))

// Human reviews and approves — load checkpoint and resume
fmt.Println("\n  [Human] Approved! Resuming graph...")
savedCP, _ := checkpointer.Load(ctx, threadID)
resumeState := savedCP.State
resumeState.Approved = true // human action

finalState, err := compiled.Invoke(ctx, resumeState, graph.WithThreadID[ApprovalState](threadID))
if err != nil {
    fmt.Println("Resume error:", err)
} else {
    fmt.Println("\nFinal result:", finalState.Result)
}

---

## 14. Parallel Branches

`AddParallelEdges` fans a node's output to multiple branches that run
concurrently. Results are merged by the `StateReducer` in arrival order
(make your reducer commutative for correctness).


In [ ]:
import (
    "context"
    "fmt"
    "sort"
    "time"

    "github.com/golangchain/golangchain/graph"
)

type ParallelState struct {
    Input   string
    Results []string
}

pg := graph.NewStateGraph(func(cur, upd ParallelState) ParallelState {
    cur.Results = append(cur.Results, upd.Results...)
    if upd.Input != "" { cur.Input = upd.Input }
    return cur
})

// fan-in node just collects
pg.MustAddNode("fanout", func(_ context.Context, s ParallelState) (ParallelState, error) {
    return ParallelState{}, nil // no-op; parallel edges kick in after
})

// Three parallel workers with simulated latency
for _, worker := range []struct{ name, tag string; delay time.Duration }{
    {"summarise", "SUMMARY",   50 * time.Millisecond},
    {"translate", "TRANSLATE", 30 * time.Millisecond},
    {"classify",  "CLASSIFY",  10 * time.Millisecond},
} {
    w := worker // capture
    pg.MustAddNode(w.name, func(_ context.Context, s ParallelState) (ParallelState, error) {
        time.Sleep(w.delay)
        return ParallelState{Results: []string{fmt.Sprintf("%s(%s)", w.tag, s.Input)}}, nil
    })
    pg.MustAddEdge(w.name, graph.END)
}

pg.MustAddEdge(graph.START, "fanout")
_ = pg.AddParallelEdges("fanout", []string{"summarise", "translate", "classify"})

pc, _ := pg.Compile()

start := time.Now()
result, _ := pc.Invoke(context.Background(), ParallelState{Input: "golangchain"})
elapsed := time.Since(start)

sort.Strings(result.Results) // deterministic output for display
fmt.Printf("Parallel branches completed in %v (sequential would take ~90ms)\n", elapsed.Round(time.Millisecond))
fmt.Println("Results:")
for _, r := range result.Results {
    fmt.Println(" ", r)
}

---

## 15. Full Pipeline: Memory + Chain + Streaming

Putting it all together: a multi-turn conversational chain with:

- `ConversationWindowMemory` for history
- `ChatPromptTemplate` with a `MessagePlaceholder`
- `LLMChain` (mock LLM) for generation
- `StrOutputParser` for clean output
- `CallbackManager` for observability
- Token-by-token streaming on the final question


In [ ]:
import (
    "context"
    "fmt"

    "github.com/golangchain/golangchain/callbacks"
    "github.com/golangchain/golangchain/chain"
    "github.com/golangchain/golangchain/memory"
    "github.com/golangchain/golangchain/output"
    "github.com/golangchain/golangchain/prompt"
    "github.com/golangchain/golangchain/schema"
)

ctx := context.Background()

// 1. Rotating mock: cycles through predefined answers.
answers := []string{
    "Amsterdam is the capital of the Netherlands.",
    "The Rijksmuseum is Amsterdam's most famous museum, housing works by Rembrandt and Vermeer.",
    "The Rijksmuseum attracts around 2.5 million visitors per year.",
    "I recommend 'The Diary of a Young Girl' by Anne Frank for its connection to Amsterdam.",
}
type rotatingMock struct{ idx int }
func (r *rotatingMock) ModelName() string { return "rotating-mock" }
func (r *rotatingMock) Generate(_ context.Context, _ []schema.Message, _ ...llm.Option) (*schema.Generation, error) {
    ans := answers[r.idx % len(answers)]
    r.idx++
    return &schema.Generation{Text: ans, Message: schema.Message{Role: schema.RoleAI, Content: ans}}, nil
}
func (r *rotatingMock) Stream(_ context.Context, _ []schema.Message, _ ...llm.Option) (<-chan schema.StreamChunk, error) {
    ch := make(chan schema.StreamChunk, 8)
    ans := answers[r.idx % len(answers)]
    r.idx++
    go func() {
        defer close(ch)
        for i, c := range ans {
            ch <- schema.StreamChunk{Text: string(c), Done: i == len(ans)-1}
        }
    }()
    return ch, nil
}
var _ llm.LLM = (*rotatingMock)(nil)

model := &rotatingMock{}

// 2. Prompt with history placeholder
chatPrompt := prompt.MustNewChatPromptTemplate(
    prompt.MustSystem("You are a knowledgeable travel assistant. Answer concisely."),
    prompt.NewMessagePlaceholder("history"),
    prompt.MustHuman("{{.question}}"),
)

// 3. Callback: count LLM calls
type counter struct {
    callbacks.NoOpHandler
    calls int
}
func (c *counter) OnLLMStart(_ context.Context, _ string, _ []schema.Message) { c.calls++ }

cnt := &counter{}
cm  := callbacks.NewCallbackManager(cnt)

// 4. LLMChain
c := chain.NewLLMChain(
    chatPrompt, model,
    output.AsAny(output.StrOutputParser{}),
    chain.WithChainName("TravelAssistant"),
    chain.WithChainCallbacks(cm),
)

// 5. Window memory (last 3 turns)
mem := memory.NewConversationWindowMemory(3)

// 6. Multi-turn conversation
questions := []string{
    "What is the capital of the Netherlands?",
    "What is its most famous museum?",
    "How many visitors does it get per year?",
}

for _, q := range questions {
    vars, _ := mem.LoadMemoryVariables(ctx)
    vars["question"] = q

    ans, _ := c.Invoke(ctx, vars)
    answer := ans.(string)
    fmt.Printf("Q: %s\nA: %s\n\n", q, answer)
    _ = mem.SaveContext(ctx, q, answer)
}

// 7. Streaming on a follow-up question
fmt.Println("--- Streaming: token by token ---")
vars, _ := mem.LoadMemoryVariables(ctx)
vars["question"] = "Can you recommend one book about Amsterdam?"

streamCh, _ := c.Stream(ctx, vars)
fmt.Print("A: ")
for chunk := range streamCh {
    if chunk.Err == nil && !chunk.Done {
        fmt.Print(chunk.Value)
    }
}
fmt.Printf("\n\nTotal LLM calls made: %d\n", cnt.calls)

---
## Connecting Real LLM Providers

Replace any `&mockLLM{...}` with a real provider — the rest of the code is
identical because all providers satisfy the same `llm.LLM` interface.

```go
// OpenAI
model, err := openai.New(
openai.WithAPIKey(os.Getenv("OPENAI_API_KEY")),
openai.WithModel("gpt-4o-mini"),
)

// Azure OpenAI
model, err := azure.New(
azure.WithAPIKey(os.Getenv("AZURE_OPENAI_API_KEY")),
azure.WithEndpoint(os.Getenv("AZURE_OPENAI_ENDPOINT")),
azure.WithDeployment("gpt-4o"),
)

// Anthropic Claude
model, err := anthropic.New(
anthropic.WithAPIKey(os.Getenv("ANTHROPIC_API_KEY")),
anthropic.WithModel("claude-sonnet-4-5"),
)

// Ollama (local)
model, err := ollama.New(
ollama.WithModel("llama3.2"),
ollama.WithBaseURL("http://localhost:11434"),
)

// Google Gemini
model, err := gemini.New(
gemini.WithAPIKey(os.Getenv("GEMINI_API_KEY")),
gemini.WithModel("gemini-2.0-flash"),
)
```
---

## Architecture Summary

```
schema          ← shared types (no imports)
  ↑
callbacks       ← 13-method Handler, CallbackManager
  ↑
llm             ← LLM interface + Options
  ↑             ↖ openai / azure / anthropic / gemini / ollama / openaicompat
prompt          ← PromptTemplate, ChatPromptTemplate, FewShot
output          ← Str / JSON / Struct / List / Bool parsers + AsAny adapter
tools           ← Tool interface, Calculator, HTTPFetch, DuckDuckGo, Shell, FuncTool
embeddings      ← Embedder interface, OpenAIEmbedder, AzureEmbedder
vectorstore     ← VectorStore interface, InMemoryVectorStore (cosine), RetrieverTool
memory          ← Buffer / Window / Summary memories
chain           ← Runnable, Pipe, LLMChain, Sequential, Map, Router
agent           ← ReActAgent, ToolCallingAgent, AgentExecutor, AgentEvent stream
graph           ← StateGraph[S], CompiledGraph[S], Interrupt, MemoryCheckpointer
```
